# 技能5 · Day 3 上机：用 deepeval 搭建营销 Agent 评测套件

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **deepeval** 为营销 Agent 搭建可运行的评测套件
2. 区分**轨迹评估**（工具调用正确性）与**端到端评估**（内容质量），分别用自定义 BaseMetric 和 GEval 实现
3. 用 **FaithfulnessMetric** 检测幻觉（输出是否忠于知识库），用 **LLM-as-a-judge** 自动评审轨迹质量
4. 用 `evaluate()` 批量运行测试套件，计算任务完成率/工具调用准确率/幻觉率

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：deepeval（confident-ai/deepeval，17k★，LLM评估框架）。
营销映射：评估营销内容生成Agent的轨迹质量（是否选对工具、是否生成合规内容）。

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> ⚠️ deepeval 默认使用 OpenAI 作为 judge 模型，需设置 `OPENAI_API_KEY` 环境变量。
> 也可配置其他模型（如 Anthropic / 本地 Ollama），见 [deepeval 文档](https://docs.confident-ai.com/)。

In [ ]:
# !pip install deepeval -q
# export OPENAI_API_KEY=<your-openai-api-key>

## 1. 数据集背景与营销映射

**评估对象**：营销内容生成 Agent 的教学合成（synthetic）/人工策展（curated）轨迹样例。我们定义3个测试用例，分别代表好/坏/混合轨迹；它们不是生产记录（recorded），不能外推生产质量。

| 用例 | 场景 | 轨迹质量 | 评估重点 |
|------|------|---------|---------|
| 用例1 | 小红书种草文案（烟酰胺精华液） | 好（工具正确、内容忠于知识库） | 端到端质量 + 幻觉检测应通过 |
| 用例2 | 朋友圈广告（新款丝绒口红） | 差（跳过搜索、虚构成分） | 工具调用准确率 + 幻觉率应报警 |
| 用例3 | 小红书种草文案（防晒霜） | 混合（工具正确但内容略有偏差） | 轨迹评估 vs 端到端评估的差异 |

每条测试用例包含：
- `input`：营销 Brief（产品+目标人群+渠道）
- `actual_output`：合成 Agent 输出内容
- `expected_output`：人工专家写的参考文案
- `retrieval_context`：知识库检索到的产品资料（用于幻觉检测）
- `trajectory`：Agent 的工具调用轨迹（用于轨迹评估，自定义属性）
- `expected_trace`：人工黄金集标注的期望工具顺序与必填参数
- `metadata`：数据等级、成本、延迟、安全标签

**营销映射**：真实项目中应替换为自己 Agent 的 curated/recorded 轨迹，并完成泄漏控制、隐私脱敏和人工黄金集校准。

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from deepeval import assert_test, evaluate
from deepeval.metrics import GEval, BaseMetric, FaithfulnessMetric
from deepeval.test_case import LLMTestCase

# deepeval 新版将 LLMTestCaseParams 重命名为 SingleTurnParams，此处兼容两种版本
try:
    from deepeval.test_case import LLMTestCaseParams
except ImportError:
    from deepeval.test_case import SingleTurnParams as LLMTestCaseParams

print("deepeval 导入完成")

## 1：准备营销Agent真实轨迹测试数据

In [ ]:
# 1. 准备营销 Agent synthetic/curated 轨迹测试数据
# 注意：这 3 条是教学合成（synthetic）并经人工策展（curated）的样例，不是生产记录（recorded）。
# CQ-S5-1 要求真实项目替换为带 provenance、泄漏控制、成本、延迟、安全标签的 curated/recorded 黄金集。

def attach_eval_metadata(test_case, expected_trace, metadata, human_labels):
    test_case.expected_trace = expected_trace
    test_case.metadata = metadata
    test_case.human_labels = human_labels
    return test_case

# 用例1：好轨迹 -- 小红书种草文案（烟酰胺精华液），工具正确、内容忠于知识库
test_case_good = LLMTestCase(
    input="为一款烟酰胺精华液写小红书种草文案，目标人群25-35岁女性",
    actual_output="姐妹们！这款烟酰胺精华液真的绝了✨ 5%浓度烟酰胺，主打提亮肤色+收缩毛孔，质地清爽不黏腻，油皮也能冲！坚持用28天，肤色明显均匀，毛孔也细腻了。价格才199元，性价比拉满！评论区扣1获取链接~ #烟酰胺 #精华液 #提亮肤色",
    expected_output="这款烟酰胺精华液含5%浓度烟酰胺，主打提亮肤色、收缩毛孔。质地清爽适合油皮。28天可见肤色均匀。售价199元。点击下方链接购买。",
    retrieval_context=[
        "产品名称：烟酰胺亮肤精华液",
        "核心成分：5%烟酰胺（提亮肤色、收缩毛孔）",
        "质地：清爽水润，适合油性肌肤",
        "建议使用周期：28天为一个见效周期",
        "售价：199元/30ml",
        "适用人群：25-35岁有肤色不均/毛孔粗大困扰的女性"
    ]
)
test_case_good.trajectory = [
    {"tool": "search_knowledge_base", "params": {"query": "烟酰胺 精华液 成分 功效"}},
    {"tool": "generate_content", "params": {"platform": "xiaohongshu", "style": "种草"}}
]
attach_eval_metadata(
    test_case_good,
    expected_trace=[
        {"tool": "search_knowledge_base", "required_params": ["query"]},
        {"tool": "generate_content", "required_params": ["platform", "style"]}
    ],
    metadata={"data_level": "synthetic/curated", "prompt_tokens": 420, "completion_tokens": 135, "estimated_cost_usd": 0.011, "latency_ms": 6200, "safety_flags": []},
    human_labels={"content_pass": True, "faithful": True, "tool_trace_pass": True, "safety_pass": True}
)

# 用例2：坏轨迹 -- 朋友圈广告（丝绒口红），跳过搜索、虚构成分
test_case_bad = LLMTestCase(
    input="为新款丝绒口红写朋友圈广告",
    actual_output="新品丝绒口红上市！富含玻尿酸精华，滋润不拔干，持久保湿8小时！一抹显白，8色可选，限时5折只要99元！快来抢购吧！",
    expected_output="新款丝绒口红，哑光丝绒质地，主打持久不脱色，8色可选。原价198元，现价99元。",
    retrieval_context=[
        "产品名称：丝绒哑光口红",
        "质地：哑光丝绒（非保湿型，主打持久不脱色）",
        "颜色：8色可选",
        "售价：198元（无折扣活动）",
        "注意：本品为哑光质地，不含保湿/滋润成分"
    ]
)
test_case_bad.trajectory = [
    {"tool": "generate_content", "params": {"platform": "weibo"}}
]
attach_eval_metadata(
    test_case_bad,
    expected_trace=[
        {"tool": "search_knowledge_base", "required_params": ["query"]},
        {"tool": "generate_content", "required_params": ["platform", "style"]}
    ],
    metadata={"data_level": "synthetic/curated", "prompt_tokens": 300, "completion_tokens": 86, "estimated_cost_usd": 0.007, "latency_ms": 3800, "safety_flags": ["unsupported_claim", "false_promotion"]},
    human_labels={"content_pass": False, "faithful": False, "tool_trace_pass": False, "safety_pass": False}
)

# 用例3：混合轨迹 -- 小红书种草文案（防晒霜），工具正确但内容略有偏差
test_case_mixed = LLMTestCase(
    input="为防晒霜写小红书种草文案",
    actual_output="夏天必备！这款防晒霜SPF50+ PA++++，防晒黑效果一流~ 质地轻薄不闷痘，防水8小时，海边玩水也不怕！一瓶50ml只要159元，快冲！#防晒 #夏日必备",
    expected_output="这款防晒霜SPF50+ PA++++，主打防晒黑，质地轻薄不闷痘。50ml售价159元。",
    retrieval_context=[
        "产品名称：清透防晒霜",
        "防晒指数：SPF50+ PA++++（主打防晒黑）",
        "质地：轻薄透气，不闷痘",
        "容量与售价：50ml / 159元",
        "注意：防水时间为4小时（非8小时）"
    ]
)
test_case_mixed.trajectory = [
    {"tool": "search_knowledge_base", "params": {"query": "防晒霜 SPF 防晒黑"}},
    {"tool": "generate_content", "params": {"platform": "xiaohongshu", "style": "种草"}}
]
attach_eval_metadata(
    test_case_mixed,
    expected_trace=[
        {"tool": "search_knowledge_base", "required_params": ["query"]},
        {"tool": "generate_content", "required_params": ["platform", "style"]}
    ],
    metadata={"data_level": "synthetic/curated", "prompt_tokens": 390, "completion_tokens": 118, "estimated_cost_usd": 0.010, "latency_ms": 5400, "safety_flags": ["unsupported_claim"]},
    human_labels={"content_pass": True, "faithful": False, "tool_trace_pass": True, "safety_pass": False}
)
# 注意：用例3工具调用正确，但 actual_output 中"防水8小时"与知识库"4小时"不符 -> 幻觉/unsupported claim

print(f"用例1（好轨迹）输出长度: {len(test_case_good.actual_output)} 字")
print(f"用例2（坏轨迹）输出长度: {len(test_case_bad.actual_output)} 字")
print(f"用例3（混合轨迹）输出长度: {len(test_case_mixed.actual_output)} 字")
print("数据等级:", {tc.metadata["data_level"] for tc in [test_case_good, test_case_bad, test_case_mixed]})

## 2. 轨迹评估 vs 端到端评估

Agent 评估的核心洞察：**不能只看答案对不对，必须评估过程好不好**。

```
端到端评估（End-to-End）：
  Brief -> [Agent 黑盒] -> 文案 -> 对比参考文案 -> 好/差

轨迹评估（Trajectory）：
  Brief -> Thought -> Action(搜索知识库) -> Obs -> Thought -> Action(生成文案) -> 文案
                 ↑                    ↑                              ↑
           推理合理？           工具选对？参数对？                步骤冗余？
```

- **端到端**：用 GEval 让 LLM-as-judge 评估内容质量（品牌调性、CTA、平台适配）--粗粒度
- **轨迹**：用自定义 BaseMetric 评估工具调用正确性（选对工具？参数正确？）--细粒度
- **幻觉**：用 FaithfulnessMetric 对比 actual_output 与 retrieval_context --检测虚构成分

两层都要：端到端做基础门禁，轨迹做深度诊断。

## 2-3：端到端评估 + 轨迹评估

In [ ]:
# 2. 端到端评估 -- 用 GEval 评估营销内容质量
content_quality_metric = GEval(
    name="营销内容质量",
    criteria="评估营销内容的质量，按以下标准打分：1)品牌调性一致性（专业美妆品牌调性）"
             " 2)CTA明确性（是否有清晰的行动号召）3)平台适配性（是否符合小红书/朋友圈风格）"
             " 4)情感共鸣度（是否能引起目标用户共鸣）5)信息准确性（产品信息是否正确）",
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT
    ],
    threshold=0.7,
    strict_mode=False,
    async_mode=False
)

for i, tc in enumerate([test_case_good, test_case_bad, test_case_mixed], 1):
    content_quality_metric.measure(tc)
    print(f"用例{i} 内容质量: {content_quality_metric.score:.2f} | {content_quality_metric.reason[:100]}")

In [ ]:
# 3. 轨迹评估 -- 自定义 BaseMetric 评估工具调用正确性
class ToolCallAccuracyMetric(BaseMetric):
    """按人工黄金集 expected_trace 评估：工具顺序 + 必填参数 + 冗余调用。"""

    def __init__(self, threshold=0.7):
        self.threshold = threshold
        self.score = 0.0
        self.reason = "未运行"
        self.success = False

    def measure(self, test_case: LLMTestCase) -> float:
        trajectory = getattr(test_case, 'trajectory', [])
        expected_trace = getattr(test_case, 'expected_trace', [])
        if not trajectory or not expected_trace:
            self.score = 0.0
            self.reason = "缺少 trajectory 或 expected_trace，无法评估工具调用"
            self.success = False
            return self.score

        max_len = max(len(trajectory), len(expected_trace))
        step_scores = []
        reasons = []
        for index in range(max_len):
            actual = trajectory[index] if index < len(trajectory) else None
            expected = expected_trace[index] if index < len(expected_trace) else None
            if actual is None:
                step_scores.append(0.0)
                reasons.append(f"step {index + 1}: 缺失期望工具 {expected['tool']}")
                continue
            if expected is None:
                step_scores.append(0.0)
                reasons.append(f"step {index + 1}: 冗余调用 {actual.get('tool')}")
                continue

            tool_score = 1.0 if actual.get('tool') == expected.get('tool') else 0.0
            params = actual.get('params', {}) or {}
            required = expected.get('required_params', [])
            missing = [name for name in required if name not in params or params.get(name) in (None, "")]
            param_score = 1.0 if not missing else 0.0
            step_score = 0.7 * tool_score + 0.3 * param_score
            step_scores.append(step_score)

            if tool_score < 1.0:
                reasons.append(f"step {index + 1}: 工具应为 {expected.get('tool')}，实际 {actual.get('tool')}")
            if missing:
                reasons.append(f"step {index + 1}: 缺失必填参数 {missing}")

        self.score = sum(step_scores) / len(step_scores)
        self.reason = "; ".join(reasons) if reasons else f"全部 {len(expected_trace)} 个期望步骤均匹配"
        self.success = self.score >= self.threshold
        return self.score

    async def a_measure(self, test_case: LLMTestCase) -> float:
        return self.measure(test_case)

    def is_successful(self) -> bool:
        if hasattr(self, 'error') and self.error is not None:
            self.success = False
        return self.success

    @property
    def __name__(self):
        return "Tool Call Accuracy"

tool_metric = ToolCallAccuracyMetric(threshold=0.7)
for i, tc in enumerate([test_case_good, test_case_bad, test_case_mixed], 1):
    tool_metric.measure(tc)
    print(f"用例{i} 工具调用准确率: {tool_metric.score:.2f} | {tool_metric.reason}")

## 3. 幻觉检测：为什么营销 Agent 必须做

营销 Agent 的幻觉后果严重：
- **虚构成分/功效**：如把"哑光口红"说成"含玻尿酸滋润"（产品实际不含）-> 违反广告法
- **虚构价格/优惠**：如编造"限时5折"（实际无此活动）-> 虚假宣传
- **虚构认证**：如编造"皮肤科医生推荐"（实际无此背书）-> 误导消费者

**FaithfulnessMetric 的工作原理**：
1. 从 `actual_output` 中提取所有事实性声明（claims）
2. 逐条与 `retrieval_context`（知识库）交叉验证
3. 忠实度 = 忠实声明数 / 总声明数
4. 低于 threshold 的用例 = 存在幻觉

这是 deepeval 内置指标，无需手写规则匹配。

## 4-5：幻觉检测 + LLM-as-a-judge自动评审

In [ ]:
# 4. 幻觉检测 -- 用 FaithfulnessMetric 检测输出是否忠于知识库
faithfulness_metric = FaithfulnessMetric(
    threshold=0.7,
    include_reason=True,
    async_mode=False
)

for i, tc in enumerate([test_case_good, test_case_bad, test_case_mixed], 1):
    faithfulness_metric.measure(tc)
    print(f"用例{i} 忠实度: {faithfulness_metric.score:.2f} | {faithfulness_metric.reason[:100]}")

print()
print("解读：")
print("  用例1（好轨迹）：忠实度应高 -- 内容忠于知识库")
print("  用例2（坏轨迹）：忠实度应低 -- 虚构了'玻尿酸精华''保湿8小时'（知识库明确说不含保湿）")
print("  用例3（混合轨迹）：忠实度可能偏低 -- '防水8小时'与知识库'4小时'不符 -> 幻觉")

In [ ]:
# 5. LLM-as-a-judge 自动评审 -- 用 GEval criteria 评估轨迹质量
# 关键修正：judge 必须看到序列化后的 trajectory，而不是只看最终 actual_output。
def format_trace_for_judge(test_case):
    actual_steps = []
    for index, step in enumerate(getattr(test_case, 'trajectory', []), 1):
        actual_steps.append(f"step {index}: tool={step.get('tool')} params={step.get('params', {})}")
    expected_steps = []
    for index, step in enumerate(getattr(test_case, 'expected_trace', []), 1):
        expected_steps.append(f"step {index}: expected_tool={step.get('tool')} required_params={step.get('required_params', [])}")
    return "\n".join([
        "ACTUAL_TRAJECTORY:",
        "\n".join(actual_steps),
        "EXPECTED_TRACE:",
        "\n".join(expected_steps),
        "FINAL_OUTPUT:",
        test_case.actual_output,
    ])

def make_trajectory_judge_case(test_case):
    return LLMTestCase(
        input=test_case.input,
        actual_output=format_trace_for_judge(test_case),
        expected_output="轨迹应匹配 expected_trace：工具顺序正确、必填参数齐全、无冗余调用；最终输出不得包含 unsupported claim。",
        retrieval_context=getattr(test_case, 'retrieval_context', [])
    )

trajectory_judge_metric = GEval(
    name="轨迹质量（LLM-as-a-judge）",
    criteria="评估Agent执行轨迹的质量，按以下标准打分："
             "1)工具选择是否匹配 EXPECTED_TRACE "
             "2)必填参数是否齐全 "
             "3)是否有冗余步骤 "
             "4)FINAL_OUTPUT 是否含 unsupported claim "
             "5)理由必须引用具体 step，不接受 good/nice 等空洞词。",
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT
    ],
    threshold=0.7,
    strict_mode=False,
    async_mode=False
)

for i, tc in enumerate([test_case_good, test_case_bad, test_case_mixed], 1):
    judge_case = make_trajectory_judge_case(tc)
    trajectory_judge_metric.measure(judge_case)
    print(f"用例{i} 轨迹质量(LLM-judge): {trajectory_judge_metric.score:.2f} | {trajectory_judge_metric.reason[:100]}")

print()
print("注意：LLM-as-a-judge 现在看到 ACTUAL_TRAJECTORY + EXPECTED_TRACE + FINAL_OUTPUT，")
print("但仍需人工黄金集、重复评估、位置偏差/长度偏差检查后才可进入 CI 阻断。")

## 5. 综合评估指标

完成 TODO 1-5 后，我们有了3个维度 × 3个用例的评估结果。TODO 6 将用 `evaluate()` 批量运行，并汇总为三个核心指标：

| 指标 | 定义 | 计算方式 | 营销 Agent 目标 |
|------|------|---------|----------------|
| 任务完成率 | 内容质量达标的用例比例 | GEval score ≥ threshold 的用例数 / 总用例数 | ≥ 85% |
| 工具调用准确率 | 工具调用正确的比例 | ToolCallAccuracyMetric score 的均值 | ≥ 90% |
| 幻觉率 | 存在幻觉的用例比例 | FaithfulnessMetric score < threshold 的用例数 / 总用例数 | ≤ 5% |

`evaluate()` 会批量运行所有 test_case × metric 组合，返回结构化结果。

## 6：综合评估（evaluate批量运行）

In [ ]:
# 6. 综合评估 -- 用 evaluate 批量运行，计算三个核心指标
all_test_cases = [test_case_good, test_case_bad, test_case_mixed]
all_metrics = [
    GEval(
        name="营销内容质量",
        criteria="评估营销内容质量：品牌调性/CTA/平台适配/情感共鸣/信息准确",
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
        threshold=0.7,
        async_mode=False
    ),
    ToolCallAccuracyMetric(threshold=0.7),
    FaithfulnessMetric(threshold=0.7, include_reason=True, async_mode=False)
]

# 用 evaluate 批量运行（返回结构化结果）
results = evaluate(
    test_cases=all_test_cases,
    metrics=all_metrics,
    print_results=False
)

# 手动计算三个综合指标
# 重新 measure 以获取每个用例的各维度分数
content_scores = []
tool_scores = []
faith_scores = []

for tc in all_test_cases:
    cm = GEval(
        name="营销内容质量",
        criteria="评估营销内容质量：品牌调性/CTA/平台适配/情感共鸣/信息准确",
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
        threshold=0.7,
        async_mode=False
    )
    cm.measure(tc)
    content_scores.append(cm.score)

    tm = ToolCallAccuracyMetric(threshold=0.7)
    tm.measure(tc)
    tool_scores.append(tm.score)

    fm = FaithfulnessMetric(threshold=0.7, include_reason=True, async_mode=False)
    fm.measure(tc)
    faith_scores.append(fm.score)

# 任务完成率 = 内容质量 >= 0.7 的用例比例
task_completion_rate = sum(1 for s in content_scores if s >= 0.7) / len(content_scores)
# 工具调用准确率 = 工具调用准确率的均值
tool_accuracy_rate = sum(tool_scores) / len(tool_scores)
# 幻觉率 = 忠实度 < 0.7 的用例比例（忠实度低 = 有幻觉）
hallucination_rate = sum(1 for s in faith_scores if s < 0.7) / len(faith_scores)

print("=" * 50)
print(f"任务完成率: {task_completion_rate:.1%}")
print(f"工具调用准确率: {tool_accuracy_rate:.1%}")
print(f"幻觉率: {hallucination_rate:.1%}")
print("=" * 50)
print()
print("各用例详细评分：")
for i, (c, t, f) in enumerate(zip(content_scores, tool_scores, faith_scores), 1):
    print(f"  用例{i}: 内容质量={c:.2f} | 工具准确率={t:.2f} | 忠实度={f:.2f}")
print()
print("解读：")
print("  - 任务完成率低 -> Agent 内容质量不达标，需优化 prompt 或知识库")
print("  - 工具调用准确率低 -> Agent 工具选择/参数有问题，需优化工具描述或路由逻辑")
print("  - 幻觉率高 -> Agent 输出不忠于知识库，需加强 RAG 检索或添加事实约束")

## 6. 反思与前沿

### 反思问题
1. 你的营销 Agent 在哪个评估维度表现最差？根因是什么（工具选择/参数/推理/幻觉）？
2. 用例2（坏轨迹）的工具调用准确率为0%，但端到端内容质量评分可能不是0--为什么？（提示：LLM 可能生成"看起来合理"但实际有幻觉的内容）
3. 如果 Agent 在测试集上表现好，但在生产中表现差，可能是什么原因？（提示：测试集覆盖不足/长尾问题/分布漂移）
4. LLM-as-a-judge 的评分本身是否可信？如何校准？（提示：人工抽检 + 多 judge 投票）

### 2026 前沿：LLM-as-a-judge + deepeval 可运行评测框架
把 LLM-as-a-judge（NeurIPS 2023, arXiv 2306.05685）写成 deepeval 的 GEval 测试用例，用 `assert_test` 断言 + `deepeval test run` 在 CI 中自动执行：
- 每次模型升级/prompt修改后，自动运行完整测试集
- 评分低于 threshold 的用例自动 fail，防止回归
- LLM-as-judge 的评分+理由结构化存储，支持按时间/场景聚合分析

**注意**：LLM-as-a-judge 是辅助评估，有自身偏差。对应因果阶梯 L1（对轨迹文本的关联分析），不能替代真实业务指标（L2 A/B测试）。定位为"开发期自检工具"。

参考 [arXiv 2306.05685](https://arxiv.org/abs/2306.05685)（NeurIPS 2023, LLM-as-a-judge）+ [deepeval](https://github.com/confident-ai/deepeval)。